---
title: "Chat with Parliament"
author: "Chresten R. Søndergaard"
title-block-banner: true
title-block-style: default
date: "2026-02-27"
abstract: >
    Here we will demonstrate *agentic delegation* where a central AI agent can delegate sub-tasks to other agents for a well-defined division of the work. While more complex than having a single AI agent it can lead to better overall performance.

    Here we set up a political analyst agent have access to a series party expert agents (one for each party). The political analyst takes input from the user about their viewpoints and then queries relevant parties on their opinions in order to rank the parties according to their alignment with the user.
    
    This page is part of the [chat_with_parliament](https://github.com/crs17/chat_with_parliament) repository which contains all the code for the demo.
---

# Chat with parliament
Here we will demonstrate a multi- AI agent implementation. A political analyst agent takes input from users about their political views and which subjects matter to them.

For each part in the Danish parliament, a party expert Agent is implement with with access to a vector database collection containing the relevant party's manifesto. Under the hood there is actually only one party expert agent, it just filters the vector database according to which party it is asked about.

In a agentic delegation setup, the political analyst can choose to query one or more of the party expert agents on specific subjects. After having gather the relevant responses from the party experts, the political analyst agent can score each of the parties in relation to their alignment with the users input.

![Architecture](static/images/chat_with_parliament.drawio.svg)


## Running the agent setup
Let's start by running the setup with two statements in opposite directions valuing either climate or economic growth the highest.

### Opinion: "Klimaet er vigtigere end erhvervslivet."

Below we see the first the logs of the model run showing how the political analyst agent queries the individual party expert agents, the responses being sent back, and finally how political analyst compiles the answers. Further below we see the scores given to the individual parties by the political analyst. This seems to align pretty well with how I would have done it intuitively. The general trend is left-wing parties at the top, right-wing parties at the bottom. 

In [1]:
!make setup > /dev/null 2>&1
!make fetch_models > /dev/null 2>&1

In [2]:
from backend.agents import political_analyst_agent


def print_party_scores(result):
    print('Party Score Reason')
    for recommendation in sorted(result.output.results, key=lambda x: x.score, reverse=True):
        print(f'{recommendation.party_id:5} {recommendation.score:5} {recommendation.reason}')


Logfire project URL: ]8;id=779093;https://logfire-eu.pydantic.dev/crs17/chat-with-parliament\https://logfire-eu.pydantic.dev/crs17/chat-with-parliament]8;;\

In [ ]:

result_klima_1 = await political_analyst_agent.run(
    "jeg mener klimaet er vigtigere end erhvervslivet! Lad mig se en score for alle partierne"
)

12:11:18.845 political_analyst_agent run
12:11:18.854   chat qwen3:8b
12:12:31.627   running 11 tools
12:12:31.636     running tool: consult_party_expert

>>> Analyst → Party Expert: Socialdemokratiet (A), query='klimapolitik'
12:12:31.650       party_expert_agent run
12:12:31.652     running tool: consult_party_expert

>>> Analyst → Party Expert: Radikale Venstre (B), query='klimapolitik'
12:12:31.657       party_expert_agent run
12:12:31.657     running tool: consult_party_expert

>>> Analyst → Party Expert: Det Konservative Folkeparti (C), query='klimapolitik'
12:12:31.658       party_expert_agent run
12:12:31.660     running tool: consult_party_expert

>>> Analyst → Party Expert: Borgernes Parti - Lars Boje Mathiesen (H), query='klimapolitik'
12:12:31.662       party_expert_agent run
12:12:31.663     running tool: consult_party_expert

>>> Analyst → Party Expert: Liberal Alliance (I), query='klimapolitik'
12:12:31.663       party_expert_agent run
12:12:31.664     running tool: cons

In [4]:
print_party_scores(result_klima_1)

Party Score Reason
Ø        10 Reelle klimamål, CO₂-afgift og bindende politik, men med økonomisk balance
Å        10 Overdagens ambitiøse mål, borgernes indflydelse og grøn økonomi
B         9 Ambitiose klimamål og klimalov, men kritik af grøn omstilling
V         9 Ambitiose mål og grøn omstilling, men balancerer klima og økonomi
A         8 Klimamål og samarbejde med erhvervslivet, men økonomisk fokus fremhæves
I         7 Balanceret tilgang med vedvarende energi og kernekraft, men ikke bindende mål
Æ         5 Kritik af EU-klimapolitik, men støtte til grøn omstilling med dansk fokus
C        -5 Ingen specifikke klimapolitikker, fokus på økonomi
M        -6 Ingen specifikke klimapolitikker, fokus på markedsbaserede løsninger
H        -7 Modstand mod grønne afgifter, prioriterer økonomi over klima
O        -8 Støtte til fossile brændsler og modstand mod grønne afgifter


### Opinion: "Økonomisk vækst er førsteprioritet."
Next, we try roughly the opposite oppinon, namely that economic growth is the top priority.

In [8]:
result_klima_2 = await political_analyst_agent.run(
    "Økonomisk vækst er førsteprioritet for mig. Lad mig se en score for alle partierne"
)

13:19:00.590 political_analyst_agent run
13:19:00.598   chat qwen3:8b
13:20:44.266   running 12 tools
13:20:44.272     running tool: consult_party_expert

>>> Analyst → Party Expert: Socialdemokratiet (A), query='økonomisk vækst'
13:20:44.275       party_expert_agent run
13:20:44.276     running tool: consult_party_expert

>>> Analyst → Party Expert: Radikale Venstre (B), query='økonomisk vækst'
13:20:44.277       party_expert_agent run
13:20:44.277     running tool: consult_party_expert

>>> Analyst → Party Expert: Det Konservative Folkeparti (C), query='økonomisk vækst'
13:20:44.278       party_expert_agent run
13:20:44.279     running tool: consult_party_expert

>>> Analyst → Party Expert: SF - Socialistisk Folkeparti (F), query='økonomisk vækst'
13:20:44.280       party_expert_agent run
13:20:44.280     running tool: consult_party_expert

>>> Analyst → Party Expert: Borgernes Parti - Lars Boje Mathiesen (H), query='økonomisk vækst'
13:20:44.281       party_expert_agent run
13:20:44

In [9]:
print_party_scores(result_klima_2)

Party Score Reason
H         9 Borgernes Parti er klart pro-vækst med fokus på lavere skatter, mindre bureaukrati og iværksætteri. Deres politik direkte fremmer virksomhed og innovation som kernesætter.
C         8 Det Konservative Folkeparti prioriterer deregulering, lavere skatter og et stærkt erhvervsliv som direkte drivkræfter for vækst. Deres fokus på kapital og innovation giver en klart traditionel vækstmodel.
I         7 Liberal Alliance understreger kapitalfrejelse og infrastrukturinvesteringer. Deres fokus på ansvarlig finanspolitik og teknologisk tilpasning giver en balancerede tilgang til vækst.
V         7 Venstre understreger jobskabelse og attraktiv arbejdskraft, men kombinerer det med miljømæssige mål. Deres fokus på konkurrenceevne og velfærdsøkonomi giver en delvist vækstorientering.
F         6 SF understreger grøn omstilling og teknologisk vækst, men samtidig kræver strengere finansregler. Fokus på bæredygtighed og social sikkerhed mindsker direkte støtte til traditi

## Technical considerations
During the implementation of this demo a couple of technical issues had 

## Tech Stack overview
- Playwright for fetching party manifests on party sites that are defensive against automated requests
- Docling for for parsing HTML files into markdown
- Chonkie for chunking
- Local Ollama deployment for LLM and embedding models. We do not use docker here as GPU access from docker is not available on Apple Silicon.
- Weaviate for vector database in docker
- Pydantic AI for Agent orchestration
- Logfire for monitoring and debugging of the Agents